# PCA: Dimensionality Reduction with Principal Component Analysis

Principal Component Analysis (PCA) is the most widely used linear dimensionality reduction technique. It transforms high-dimensional data into a lower-dimensional form by identifying the directions (principal components) that capture the maximum variance in the data. This notebook provides a thorough walkthrough — from intuition and math to real-world applications.

## Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_digits, fetch_olivetti_faces, make_blobs
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, KernelPCA, IncrementalPCA, SparsePCA, TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from mpl_toolkits.mplot3d import Axes3D

sns.set_style('whitegrid')
np.random.seed(42)
%matplotlib inline

---
## 1. The Curse of Dimensionality: Why Reduce Dimensions?

As the number of features grows:
- **Data becomes sparse** — points drift exponentially far apart, making distance-based methods unreliable.
- **Computational cost** increases with $O(n^3)$ for many algorithms.
- **Overfitting risk** rises — the model has too many degrees of freedom.
- **Visualisation** becomes impossible beyond 3D.

Dimensionality reduction mitigates all these issues by projecting the data into a meaningful lower-dimensional subspace.

---
## 2. Intuition: Finding the Directions of Maximum Variance

PCA finds a set of orthogonal axes (principal components) such that:
1. **PC1** points in the direction of **largest variance**.
2. **PC2** is orthogonal to PC1 and captures the **next largest variance**, and so on.

Think of PCA as rotating the coordinate system to align with the natural spread of the data. The components are purely geometric — they know nothing about class labels.

In [ ]:
# Visual intuition: 2D data with one dominant direction of variance
rng = np.random.RandomState(42)
X = rng.randn(300, 2)
X = X @ np.array([[3, 1], [0.5, 1]])  # stretch & rotate

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].scatter(X[:, 0], X[:, 1], alpha=0.6, edgecolors='k')
axes[0].set_title('Original 2D Data')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
axes[0].axis('equal')

pca_int = PCA(n_components=2)
pca_int.fit(X)
mean = pca_int.mean_
components = pca_int.components_

axes[1].scatter(X[:, 0], X[:, 1], alpha=0.6, edgecolors='k')
for i, (comp, var) in enumerate(zip(components, pca_int.explained_variance_)):
    axes[1].arrow(mean[0], mean[1], comp[0] * np.sqrt(var) * 2, comp[1] * np.sqrt(var) * 2,
                  head_width=0.2, head_length=0.3, fc=f'C{i}', ec=f'C{i}', linewidth=2.5)
axes[1].set_title('Directions of Maximum Variance')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].axis('equal')
plt.tight_layout()
plt.show()

---
## 3. PCA Step-by-Step

### Mathematical Pipeline
1. **Standardize** the data (zero mean, unit variance).
2. Compute the **covariance matrix** $\Sigma = \frac{1}{n-1} X^T X$.
3. Perform **eigendecomposition**: $\Sigma v = \lambda v$.
4. **Sort** eigenvectors by descending eigenvalues.
5. **Project** the data onto the top-$k$ eigenvectors: $X' = X W_k$.

In [ ]:
# Step-by-step PCA on a simple dataset
X_demo, _ = make_blobs(n_samples=100, n_features=4, centers=3, random_state=42)

# 1. Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_demo)

# 2. Covariance matrix
cov = np.cov(X_scaled.T)

# 3. Eigendecomposition
eigenvals, eigenvecs = np.linalg.eig(cov)

# 4. Sort descending
idx = np.argsort(eigenvals)[::-1]
eigenvals_sorted = eigenvals[idx]
eigenvecs_sorted = eigenvecs[:, idx]

# 5. Project onto top 2 components
W = eigenvecs_sorted[:, :2]
X_projected = X_scaled @ W

print(f"Covariance matrix shape: {cov.shape}")
print(f"Eigenvalues (sorted): {np.round(eigenvals_sorted, 4)}")
print(f"Projected data shape: {X_projected.shape}")

# Verify with sklearn
pca_check = PCA(n_components=2)
X_sklearn = pca_check.fit_transform(X_scaled)
print(f"\nMatches sklearn? {np.allclose(np.abs(X_projected), np.abs(X_sklearn), atol=1e-10)}")

---
## 4. Explained Variance Ratio: Choosing the Number of Components

Each principal component explains a portion of the total dataset variance. The **explained variance ratio** tells us how much information (variance) is captured by each component. We choose $k$ such that the cumulative ratio exceeds a threshold (e.g. 0.95).

In [ ]:
digits = load_digits()
X_digits, y_digits = digits.data, digits.target

scaler = StandardScaler()
X_digits_scaled = scaler.fit_transform(X_digits)

pca_full = PCA()
pca_full.fit(X_digits_scaled)

cumulative = np.cumsum(pca_full.explained_variance_ratio_)
n_95 = np.argmax(cumulative >= 0.95) + 1

print(f"Number of features: {X_digits.shape[1]}")
print(f"Components to explain 95% variance: {n_95}")
print(f"First 5 explained variance ratios: {np.round(pca_full.explained_variance_ratio_[:5], 4)}")

---
## 5. Scree Plot & Cumulative Explained Variance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scree plot
axes[0].bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
            pca_full.explained_variance_ratio_, alpha=0.7, color='steelblue')
axes[0].plot(range(1, len(pca_full.explained_variance_ratio_) + 1),
             pca_full.explained_variance_ratio_, 'ro-', markersize=3)
axes[0].axhline(y=0.05, color='grey', linestyle='--', alpha=0.5)
axes[0].set_title('Scree Plot')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')

# Cumulative explained variance
axes[1].plot(range(1, len(cumulative) + 1), cumulative, 'bo-', markersize=3)
axes[1].axhline(y=0.95, color='red', linestyle='--', label='95% threshold')
axes[1].axvline(x=n_95, color='green', linestyle='--', label=f'k={n_95}')
axes[1].set_title('Cumulative Explained Variance')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance Ratio')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 6. PCA for Visualization: Projecting High-Dim Data to 2D / 3D

PCA is often used to visualise high-dimensional datasets. Below we reduce the 64-pixel digit images to 2 and 3 dimensions.

In [ ]:
# 2D projection
pca_2d = PCA(n_components=2)
X_digits_2d = pca_2d.fit_transform(X_digits_scaled)

plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_digits_2d[:, 0], X_digits_2d[:, 1], c=y_digits,
                      cmap='tab10', alpha=0.7, edgecolors='k', linewidth=0.3)
plt.colorbar(scatter, ticks=range(10), label='Digit')
plt.title('Digits Dataset: 64D → 2D via PCA')
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.2%})')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.2%})')
plt.tight_layout()
plt.show()

In [ ]:
# 3D projection
pca_3d = PCA(n_components=3)
X_digits_3d = pca_3d.fit_transform(X_digits_scaled)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(X_digits_3d[:, 0], X_digits_3d[:, 1], X_digits_3d[:, 2],
                     c=y_digits, cmap='tab10', alpha=0.7, edgecolors='k', linewidth=0.2)
ax.set_title('Digits Dataset: 64D → 3D via PCA')
ax.set_xlabel(f'PC1 ({pca_3d.explained_variance_ratio_[0]:.2%})')
ax.set_ylabel(f'PC2 ({pca_3d.explained_variance_ratio_[1]:.2%})')
ax.set_zlabel(f'PC3 ({pca_3d.explained_variance_ratio_[2]:.2%})')
plt.colorbar(scatter, ticks=range(10), label='Digit', shrink=0.6)
plt.tight_layout()
plt.show()

---
## 7. Implementation with scikit-learn `PCA`

Scikit-learn's `PCA` class uses `scipy.linalg.svd` under the hood (LAPACK). It handles centering automatically. Configure it via:
- `n_components` — number of dimensions to keep, or a float in (0,1] for variance threshold.
- `whiten=True` — scales components to unit variance (useful for preprocessing).

In [ ]:
# Use variance threshold
pca_95 = PCA(n_components=0.95)
X_reduced = pca_95.fit_transform(X_digits_scaled)
print(f"Original: {X_digits_scaled.shape[1]} features → Reduced: {X_reduced.shape[1]} features")
print(f"Explained variance ratio sum: {pca_95.explained_variance_ratio_.sum():.4f}")

---
## 8. Implementing PCA from Scratch Using SVD

Instead of eigendecomposition of $X^T X$, we can use the **Singular Value Decomposition**:

$$X = U \Sigma V^T$$

The right singular vectors $V$ are the principal components, and the singular values $\sigma_i$ relate to the eigenvalues via $\lambda_i = \frac{\sigma_i^2}{n-1}$.

**Advantage**: SVD is numerically more stable, especially when $n_{\text{samples}} < n_{\text{features}}$.

In [ ]:
class PCAFromScratch:
    def __init__(self, n_components):
        self.n_components = n_components

    def fit(self, X):
        X_centered = X - X.mean(axis=0)
        U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
        self.components_ = Vt[:self.n_components]
        self.singular_values_ = S[:self.n_components]
        self.explained_variance_ = S[:self.n_components] ** 2 / (X.shape[0] - 1)
        self.explained_variance_ratio_ = self.explained_variance_ / (S ** 2 / (X.shape[0] - 1)).sum()
        self.mean_ = X.mean(axis=0)
        return self

    def transform(self, X):
        X_centered = X - self.mean_
        return X_centered @ self.components_.T

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)


# Test
pca_scratch = PCAFromScratch(n_components=2)
X_scratch = pca_scratch.fit_transform(X_digits_scaled)

pca_sk = PCA(n_components=2)
X_sk = pca_sk.fit_transform(X_digits_scaled)

print(f"Matches sklearn (up to sign flip): {np.allclose(np.abs(X_scratch), np.abs(X_sk), atol=1e-10)}")
print(f"Explained variance ratio (scratch): {np.round(pca_scratch.explained_variance_ratio_, 4)}")
print(f"Explained variance ratio (sklearn): {np.round(pca_sk.explained_variance_ratio_, 4)}")

---
## 9. Interpreting Principal Components (Loadings)

The **loadings** are the coefficients of the original features in each principal component. They tell us which original features contribute most to a given PC. We can visualise this as a heatmap.

In [ ]:
# Loadings heatmap for the first 4 PCs of the digits dataset
loadings = pd.DataFrame(
    pca_full.components_[:4],
    columns=[f'pixel_{i}' for i in range(64)],
    index=[f'PC{i+1}' for i in range(4)]
)

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for i, ax in enumerate(axes.flat):
    im = ax.imshow(pca_full.components_[i].reshape(8, 8), cmap='RdBu', vmin=-0.5, vmax=0.5)
    ax.set_title(f'PC{i+1} — Loadings (8×8 pixel weights)')
    ax.axis('off')
    plt.colorbar(im, ax=ax, shrink=0.7)
plt.suptitle('PCA Loadings: Pixel Contributions to First 4 Principal Components', y=1.02)
plt.tight_layout()
plt.show()

---
## 10. PCA vs t-SNE vs UMAP (Conceptual Comparison)

| Method | Type | Speed | Preserves Global Structure | Preserves Local Structure | Best For |
|--------|------|-------|---------------------------|---------------------------|----------|
| **PCA** | Linear | Very fast | ✅ Strong | ❌ Weak | First look, preprocessing, interpretability |
| **t-SNE** | Non-linear | Slow | ❌ Weak | ✅ Strong | Visualisation of clusters (2D/3D) |
| **UMAP** | Non-linear | Fast | ✅ Moderate | ✅ Strong | Visualisation & general non-linear reduction |

- **PCA** produces a linear projection that is deterministic and globally faithful.
- **t-SNE** uses probability distributions over neighbours and minimises KL divergence; excellent for revealing clusters but distances between clusters are meaningless.
- **UMAP** builds a fuzzy topological representation; often faster than t-SNE and better preserves global structure.

---
## 11. PCA for Noise Reduction (Compress and Reconstruct)

By keeping only the top-$k$ components and projecting back to the original space, we obtain a **denoised** version of the data — high-frequency noise lives in the discarded components.

In [ ]:
def reconstruct_digits(pca, X_scaled, n_components, scaler):
    pca_tmp = PCA(n_components=n_components)
    X_transformed = pca_tmp.fit_transform(X_scaled)
    X_reconstructed = pca_tmp.inverse_transform(X_transformed)
    return scaler.inverse_transform(X_reconstructed)

n_comp_list = [2, 8, 20, 40, 64]
fig, axes = plt.subplots(2, len(n_comp_list) + 1, figsize=(16, 4))

for row in range(2):
    idx = row * 4
    # Original
    axes[row, 0].imshow(digits.images[idx], cmap='gray')
    axes[row, 0].set_title('Original')
    axes[row, 0].axis('off')

    for col, n in enumerate(n_comp_list, 1):
        recon = reconstruct_digits(PCA, X_digits_scaled, n, scaler)
        axes[row, col].imshow(recon[idx].reshape(8, 8), cmap='gray')
        axes[row, col].set_title(f'{n} PCs')
        axes[row, col].axis('off')

plt.suptitle('PCA Reconstruction: Original vs Compressed-and-Restored Digits', y=1.02)
plt.tight_layout()
plt.show()

---
## 12. Using PCA as a Preprocessing Step Before ML

PCA can improve both speed and generalisation by removing redundant/noisy features before feeding data into a classifier.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_digits, y_digits, test_size=0.3, random_state=42
)

pipeline_full = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=5000, random_state=42))
])

pipeline_pca = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.95)),
    ('clf', LogisticRegression(max_iter=5000, random_state=42))
])

pipeline_full.fit(X_train, y_train)
pipeline_pca.fit(X_train, y_train)

acc_full = accuracy_score(y_test, pipeline_full.predict(X_test))
acc_pca = accuracy_score(y_test, pipeline_pca.predict(X_test))

print(f"Accuracy (full 64 features): {acc_full:.4f}")
print(f"Accuracy (PCA-reduced):      {acc_pca:.4f}")
print(f"PCA components used: {pipeline_pca.named_steps['pca'].n_components_}")

---
## 13. Kernel PCA for Non-Linear Dimensionality Reduction

Kernel PCA applies the kernel trick to find principal components in a higher-dimensional (RBF) feature space. Useful when the data manifold is non-linear.

In [ ]:
# Concentric circles (non-linearly separable)
from sklearn.datasets import make_circles

X_circ, y_circ = make_circles(n_samples=500, factor=0.3, noise=0.05, random_state=42)

kpca = KernelPCA(n_components=2, kernel='rbf', gamma=15)
X_kpca = kpca.fit_transform(X_circ)

pca_lin = PCA(n_components=2)
X_pca_lin = pca_lin.fit_transform(X_circ)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(X_circ[:, 0], X_circ[:, 1], c=y_circ, cmap='bwr', edgecolors='k', alpha=0.7)
axes[0].set_title('Original (Circles)')
axes[1].scatter(X_pca_lin[:, 0], X_pca_lin[:, 1], c=y_circ, cmap='bwr', edgecolors='k', alpha=0.7)
axes[1].set_title('Linear PCA')
axes[2].scatter(X_kpca[:, 0], X_kpca[:, 1], c=y_circ, cmap='bwr', edgecolors='k', alpha=0.7)
axes[2].set_title('Kernel PCA (RBF, γ=15)')
plt.tight_layout()
plt.show()

---
## 14. Incremental PCA for Large Datasets

Standard PCA requires the full data matrix in memory. **IncrementalPCA** processes data in mini-batches, making it feasible for datasets that do not fit in RAM.

In [ ]:
n_batches = 10
batch_size = X_digits_scaled.shape[0] // n_batches

ipca = IncrementalPCA(n_components=30)
for i in range(n_batches):
    start = i * batch_size
    end = start + batch_size
    ipca.partial_fit(X_digits_scaled[start:end])

# Compare with standard PCA
pca_batch = PCA(n_components=30)
pca_batch.fit(X_digits_scaled)

print(f"Explained variance ratio sum (batch PCA):  {pca_batch.explained_variance_ratio_.sum():.4f}")
print(f"Explained variance ratio sum (Incremental): {ipca.explained_variance_ratio_.sum():.4f}")
print(f"Components match (up to sign)? {np.allclose(np.abs(pca_batch.components_), np.abs(ipca.components_), atol=1e-4)}")

---
## 15. Real-World Example: Digits Dataset

We classify digits on both original and PCA-reduced data and compare performance across different numbers of components.

In [ ]:
n_components_range = [2, 5, 10, 15, 20, 30, 40, 50, 64]
train_accs = []
test_accs = []

for n in n_components_range:
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=n) if n < 64 else 'passthrough'),
        ('clf', LogisticRegression(max_iter=5000, random_state=42))
    ])
    if n == 64:
        # Don't apply PCA when n == 64
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(max_iter=5000, random_state=42))
        ])
    pipe.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, pipe.predict(X_train)))
    test_accs.append(accuracy_score(y_test, pipe.predict(X_test)))

best_idx = np.argmax(test_accs)
print(f"Best test accuracy: {test_accs[best_idx]:.4f} at n_components={n_components_range[best_idx]}")

plt.figure(figsize=(9, 5))
plt.plot(n_components_range, train_accs, 'bo-', label='Train accuracy')
plt.plot(n_components_range, test_accs, 'rs-', label='Test accuracy')
plt.axvline(x=n_components_range[best_idx], color='grey', linestyle='--', alpha=0.5)
plt.xlabel('Number of PCA Components')
plt.ylabel('Accuracy')
plt.title('Logistic Regression Performance vs Number of Principal Components')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 16. Sparse PCA and Truncated SVD (for Sparse / Text Data)

- **TruncatedSVD** (aka LSA) works directly on sparse matrices without centering (can't center sparse data). Ideal for TF-IDF or bag-of-words representation.
- **SparsePCA** trades off reconstruction error for sparsity in the components, making them more interpretable.

In [ ]:
from scipy.sparse import csr_matrix

# Simulate sparse text-like data
rng = np.random.RandomState(42)
X_sparse = csr_matrix(rng.binomial(1, 0.1, size=(500, 1000)))
print(f"Sparsity: {1 - X_sparse.nnz / (X_sparse.shape[0] * X_sparse.shape[1]):.2%}")

# TruncatedSVD
svd = TruncatedSVD(n_components=10, random_state=42)
X_svd = svd.fit_transform(X_sparse)
print(f"TruncatedSVD explained variance ratio: {svd.explained_variance_ratio_.sum():.4f}")

# SparsePCA
spca = SparsePCA(n_components=5, alpha=1, random_state=42)
# SparsePCA requires dense input or dense transform; we demo on dense subset
X_spca = spca.fit_transform(X_sparse[:100].toarray())
print(f"SparsePCA components shape: {spca.components_.shape}")
print(f"Non-zero entries per component: {[np.count_nonzero(c) for c in spca.components_]}")

---
## 17. Pros and Cons of PCA

### ✅ Advantages
- **Fast and scalable** — $O(\min(n^2 p, n p^2))$ with SVD; IncrementalPCA handles out-of-core data.
- **Deterministic** — no randomness, reproducible.
- **Interpretable** — loadings show feature contributions.
- **Noise reduction** — discarding low-variance components removes noise.
- **Preprocessing** — decorrelates features, useful before models sensitive to multicollinearity.

### ❌ Disadvantages
- **Linear only** — cannot capture non-linear manifolds (covered by Kernel PCA, t-SNE, UMAP).
- **Global variance** — directions are global; may miss local structure.
- **Scale-sensitive** — requires standardisation.
- **Interpretability degrades** — PCs are linear combinations of *all* original features; Sparse PCA helps.
- **Outliers** — highly sensitive; robust PCA variants exist.

---
## 18. Summary

PCA is the Swiss Army knife of dimensionality reduction. Use it for:
- Exploring and visualising high-dimensional data
- Compressing features before modelling
- Denoising via low-rank reconstruction
- Decorrelating inputs for linear models

When you need non-linear embeddings, reach for **t-SNE** (clusters in 2D) or **UMAP** (general purpose).

---
## Practice Exercises

1. **Wine dataset**: Load `sklearn.datasets.load_wine()` (13 features). Apply PCA and determine the minimum number of components needed to explain 90% of the variance. Plot the 2D projection coloured by class.

2. **Olivetti faces**: Load `fetch_olivetti_faces()`. Apply PCA and reconstruct the first 5 faces using 10, 50, 100, and 200 components. Visualise original vs reconstructed in a grid.

3. **Kernel PCA on moons**: Generate `make_moons(n_samples=500, noise=0.07)`. Apply linear PCA and kernel PCA (RBF) and compare their ability to separate the two classes in 2D.

4. **Loadings analysis**: For the digits dataset, pick the first 2 PCs and identify which pixel positions (i, j) have the highest absolute loadings. Visualise using a heatmap of the 8×8 weights.

5. **Noisy digits**: Add Gaussian noise ($\sigma=2$) to the digits data. Use PCA with 20 components to denoise the images. Plot 5 original, 5 noisy, and 5 reconstructed digits side by side.